In [ ]:
import ee
import geemap
import json
import numpy as np
import pandas as pd
from datetime import datetime

ee.Authenticate(auth_mode='localhost')
ee.Initialize(project='ee-jairsoto186')

### Bloque 1 — Setup e inicialización

In [ ]:
with open("chagres_geometry.geojson", encoding="utf-8") as f:
    gj = json.load(f)

chagres_geom = ee.Geometry(gj["features"][0]["geometry"])

FECHA_INICIO = "2000-02-01"   # MOD13A3 arranca feb 2000
FECHA_FIN    = "2026-05-01"

print("✅ Inicializado correctamente")
print(f"Rango de datos: {FECHA_INICIO} → {FECHA_FIN}")

### Bloque 2 — Extracción NDVI mensual desde GEE

In [ ]:
def extraer_ndvi_mensual(geom, start, end):
    col = (
        ee.ImageCollection("MODIS/061/MOD13A3")
        .filterBounds(geom)
        .filterDate(start, end)
        .select("NDVI")
    )

    def reducir(img):
        valor = (
            img.multiply(0.0001)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom,
                scale=1000,
                maxPixels=1e9
            )
            .get("NDVI")
        )
        return ee.Feature(None, {
            "fecha": img.date().format("YYYY-MM-dd"),
            "ndvi": valor
        })

    return col.map(reducir).getInfo()["features"]

print("Extrayendo NDVI... (puede tardar ~1 min)")
ndvi_raw = extraer_ndvi_mensual(chagres_geom, FECHA_INICIO, FECHA_FIN)

df_ndvi = pd.DataFrame([f["properties"] for f in ndvi_raw])
df_ndvi["fecha"] = pd.to_datetime(df_ndvi["fecha"])
df_ndvi = (
    df_ndvi.dropna()
    .sort_values("fecha")
    .reset_index(drop=True)
)
df_ndvi["fecha"] = df_ndvi["fecha"].dt.to_period("M").dt.to_timestamp()

df_ndvi.to_csv("ndvi_mensual_chagres_2000.csv", index=False)
print(f"✅ {len(df_ndvi)} registros NDVI guardados")
df_ndvi.tail()

In [ ]:
# MOD11A2 es cada 8 días, así que promediamos por mes

def extraer_lst_mensual(geom, start, end):
    col = (
        ee.ImageCollection("MODIS/061/MOD11A2")
        .filterBounds(geom)
        .filterDate(start, end)
        .select("LST_Day_1km")
    )

    def reducir(img):
        valor = (
            img.multiply(0.02).subtract(273.15)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom,
                scale=1000,
                maxPixels=1e9
            )
            .get("LST_Day_1km")
        )
        return ee.Feature(None, {
            "fecha": img.date().format("YYYY-MM-dd"),
            "lst_celsius": valor
        })

    return col.map(reducir).getInfo()["features"]

print("Extrayendo LST... (puede tardar ~2 min)")
lst_raw = extraer_lst_mensual(chagres_geom, FECHA_INICIO, FECHA_FIN)

df_lst = pd.DataFrame([f["properties"] for f in lst_raw])
df_lst["fecha"] = pd.to_datetime(df_lst["fecha"])
df_lst = df_lst.dropna().sort_values("fecha").reset_index(drop=True)

# Agregar a mensual (promedio de las lecturas de 8 días dentro del mes)
df_lst["fecha"] = df_lst["fecha"].dt.to_period("M").dt.to_timestamp()
df_lst = df_lst.groupby("fecha")["lst_celsius"].mean().reset_index()

df_lst.to_csv("lst_mensual_chagres_2000.csv", index=False)
print(f"✅ {len(df_lst)} registros LST guardados")
df_lst.tail()

### Bloque 4 — Extracción precipitación CHIRPS mensual


In [ ]:
def extraer_precip_era5land(geom, start, end):
    col = (
        ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
        .filterBounds(geom)
        .filterDate(start, end)
        .select("total_precipitation_sum")
    )

    def reducir(img):
        valor = (
            img.multiply(1000)  # metros → milímetros
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom,
                scale=9000,
                maxPixels=1e9
            )
            .get("total_precipitation_sum")
        )
        return ee.Feature(None, {
            "fecha": img.date().format("YYYY-MM-dd"),
            "precip_mm": valor
        })

    return col.map(reducir).getInfo()["features"]

print("Extrayendo precipitación ERA5-Land mensual...")
precip_raw = extraer_precip_era5land(chagres_geom, FECHA_INICIO, FECHA_FIN)

df_precip = pd.DataFrame([f["properties"] for f in precip_raw])
df_precip["fecha"] = pd.to_datetime(df_precip["fecha"])
df_precip = (
    df_precip.dropna()
    .sort_values("fecha")
    .reset_index(drop=True)
)
df_precip["fecha"] = df_precip["fecha"].dt.to_period("M").dt.to_timestamp()

# Verificar antes de guardar
print(f"Rango: {df_precip['fecha'].min().date()} → {df_precip['fecha'].max().date()}")
print(f"Filas: {len(df_precip)}")
print(df_precip.tail())

# Solo guardar si llega a 2026
if df_precip["fecha"].max().year >= 2025:
    df_precip.to_csv("precip_mensual_chagres_2000.csv", index=False)
    print("✅ CSV guardado correctamente")
else:
    print("❌ Algo falló, el CSV no se sobreescribió")

In [ ]:
def extraer_precip_era5land(geom, start, end):
    col = (
        ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
        .filterBounds(geom)
        .filterDate(start, end)
        .select("total_precipitation_sum")
    )

    def reducir(img):
        valor = (
            img.multiply(1000)  # metros → milímetros
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom,
                scale=9000,
                maxPixels=1e9
            )
            .get("total_precipitation_sum")
        )
        return ee.Feature(None, {
            "fecha": img.date().format("YYYY-MM-dd"),
            "precip_mm": valor
        })

    return col.map(reducir).getInfo()["features"]

print("Extrayendo precipitación ERA5-Land mensual...")
precip_raw = extraer_precip_era5land(chagres_geom, FECHA_INICIO, FECHA_FIN)

df_precip = pd.DataFrame([f["properties"] for f in precip_raw])
df_precip["fecha"] = pd.to_datetime(df_precip["fecha"])
df_precip = (
    df_precip.dropna()
    .sort_values("fecha")
    .reset_index(drop=True)
)
df_precip["fecha"] = df_precip["fecha"].dt.to_period("M").dt.to_timestamp()

# Verificar antes de guardar
print(f"Rango: {df_precip['fecha'].min().date()} → {df_precip['fecha'].max().date()}")
print(f"Filas: {len(df_precip)}")
print(df_precip.tail())

# Solo guardar si llega a 2026
if df_precip["fecha"].max().year >= 2025:
    df_precip.to_csv("precip_mensual_chagres_2000.csv", index=False)
    print("✅ CSV guardado correctamente")
else:
    print("❌ Algo falló, el CSV no se sobreescribió")

### Bloque 5 — ONI desde NOAA


In [ ]:
# Descargamos el ONI oficial de NOAA y lo convertimos a formato largo
# -99.9 = dato faltante (meses futuros sin pronóstico confirmado)

url_oni = "https://psl.noaa.gov/data/correlation/oni.data"

df_oni_raw = pd.read_csv(
    url_oni,
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=["year","ene","feb","mar","abr","may","jun",
           "jul","ago","sep","oct","nov","dic"]
)
df_oni_raw = df_oni_raw.apply(pd.to_numeric, errors="coerce").dropna()
df_oni_raw["year"] = df_oni_raw["year"].astype(int)
df_oni_raw = df_oni_raw[
    (df_oni_raw["year"] >= 2000) &
    (df_oni_raw["year"] <= 2030) &
    (df_oni_raw["ene"] > -90)
]

meses_map = {
    "ene":1, "feb":2, "mar":3, "abr":4, "may":5, "jun":6,
    "jul":7, "ago":8, "sep":9, "oct":10, "nov":11, "dic":12
}

oni_records = []
for _, row in df_oni_raw.iterrows():
    year = int(row["year"])
    for mes_str, mes_num in meses_map.items():
        val = row[mes_str]
        if val > -90:
            oni_records.append({
                "fecha": pd.Timestamp(year=year, month=mes_num, day=1),
                "oni": val
            })

df_oni = (
    pd.DataFrame(oni_records)
    .sort_values("fecha")
    .reset_index(drop=True)
)

df_oni.to_csv("oni_mensual_2000.csv", index=False)
print(f"✅ {len(df_oni)} registros ONI guardados")
print(df_oni.tail(8))

### Bloque 6 — Construcción del dataset maestro


In [ ]:
# Cargar los CSVs por si reinicias el kernel
df_ndvi   = pd.read_csv("ndvi_mensual_chagres_2000.csv",  parse_dates=["fecha"])
df_lst    = pd.read_csv("lst_mensual_chagres_2000.csv",   parse_dates=["fecha"])
df_precip = pd.read_csv("precip_mensual_chagres_2000.csv", parse_dates=["fecha"])
df_oni    = pd.read_csv("oni_mensual_2000.csv",           parse_dates=["fecha"])

# Normalizar todas las fechas al primer día del mes
for df in [df_ndvi, df_lst, df_precip, df_oni]:
    df["fecha"] = df["fecha"].dt.to_period("M").dt.to_timestamp()

# Merge por fecha
df = (
    df_ndvi
    .merge(df_lst,    on="fecha", how="inner")
    .merge(df_precip, on="fecha", how="inner")
    .merge(df_oni,    on="fecha", how="inner")
    .sort_values("fecha")
    .reset_index(drop=True)
)

print(f"Dataset combinado: {len(df)} filas")
print(f"Rango: {df['fecha'].min()} → {df['fecha'].max()}")
print(f"\nNulos:\n{df.isnull().sum()}")
df.head()

### Bloque 7 — Feature engineering sin data leakage
 

In [ ]:
# Aquí está la clave que faltaba antes:
# Al predecir el NDVI del mes siguiente, solo podemos usar
# información disponible ANTES de ese mes.
# Por eso todo va con lag mínimo de 1 mes.

# Target: NDVI del mes siguiente
df["ndvi_target"] = df["ndvi"].shift(-1)

# Lags de NDVI (información pasada del bosque)
df["ndvi_lag1"] = df["ndvi"].shift(1)
df["ndvi_lag2"] = df["ndvi"].shift(2)
df["ndvi_lag3"] = df["ndvi"].shift(3)

# NDVI promedio móvil 3 meses (tendencia reciente)
df["ndvi_rolling3"] = df["ndvi"].shift(1).rolling(3).mean()

# Lags de ONI — el efecto del Niño tiene delay de 1-3 meses en vegetación
df["oni_lag1"] = df["oni"].shift(1)
df["oni_lag2"] = df["oni"].shift(2)
df["oni_lag3"] = df["oni"].shift(3)

# ONI promedio 3 meses (intensidad acumulada del evento)
df["oni_rolling3"] = df["oni"].shift(1).rolling(3).mean()

# LST y precipitación con lag 1 (no los tenemos "hoy" para predecir mañana)
df["lst_lag1"]    = df["lst_celsius"].shift(1)
df["precip_lag1"] = df["precip_mm"].shift(1)

# Estacionalidad cíclica (el mes del año captura patrones secos/lluviosos)
df["mes"] = df["fecha"].dt.month
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

# Clasificación del estado ENSO según ONI estándar
# (útil como feature categórica)
def clasificar_enso(oni):
    if oni >= 0.5:
        return 1    # El Niño
    elif oni <= -0.5:
        return -1   # La Niña
    else:
        return 0    # Neutro

df["enso_estado"] = df["oni_lag1"].apply(clasificar_enso)

# Eliminar filas con NaN (primeros meses por los lags, último por el target)
df_modelo = df.dropna().reset_index(drop=True)

print(f"Dataset final: {len(df_modelo)} filas")
print(f"Rango: {df_modelo['fecha'].min()} → {df_modelo['fecha'].max()}")
df_modelo[["fecha","ndvi","oni","ndvi_lag1","oni_lag1","ndvi_target"]].head(8)

### Calcular promedio histórico por mes del año (climatología)

In [ ]:
# Calcular promedio histórico por mes del año (climatología)
df_modelo["mes"] = df_modelo["fecha"].dt.month

climatologia = (
    df_modelo.groupby("mes")["ndvi"]
    .mean()
    .rename("ndvi_climatologia")
)

df_modelo = df_modelo.merge(climatologia, on="mes", how="left")

# Anomalía = NDVI observado - promedio histórico de ese mes
df_modelo["ndvi_anomalia"]          = df_modelo["ndvi"] - df_modelo["ndvi_climatologia"]
df_modelo["ndvi_anomalia_target"]   = df_modelo["ndvi_target"] - df_modelo["ndvi_climatologia"].shift(-1)

# Lags de anomalía en vez de NDVI absoluto
df_modelo["anom_lag1"] = df_modelo["ndvi_anomalia"].shift(1)
df_modelo["anom_lag2"] = df_modelo["ndvi_anomalia"].shift(2)
df_modelo["anom_lag3"] = df_modelo["ndvi_anomalia"].shift(3)
df_modelo["anom_rolling3"] = df_modelo["ndvi_anomalia"].shift(1).rolling(3).mean()

df_modelo = df_modelo.dropna().reset_index(drop=True)

print("Anomalías calculadas")
print(f"Rango anomalía target: {df_modelo['ndvi_anomalia_target'].min():.4f} → {df_modelo['ndvi_anomalia_target'].max():.4f}")
print(f"\nDistribución anomalía target:")
print(df_modelo["ndvi_anomalia_target"].describe().round(4))

### Convertir a problema de clasificación binaria

In [ ]:
# Convertir a problema de clasificación binaria
# 1 = estrés (anomalía negativa, NDVI bajo la media estacional)
# 0 = normal o por encima

df_modelo["estres_target"] = (df_modelo["ndvi_anomalia_target"] < 0).astype(int)

# Ver distribución de clases
print("Distribución de clases:")
print(df_modelo["estres_target"].value_counts())
print(f"\n% meses con estrés: {df_modelo['estres_target'].mean()*100:.1f}%")

# Ver cuántos meses de estrés coinciden con El Niño
tabla = pd.crosstab(
    df_modelo["estres_target"],
    df_modelo["oni_lag1"].apply(lambda x: "El Niño" if x >= 0.5 else ("La Niña" if x <= -0.5 else "Neutro")),
    margins=True
)
print("\nEstrés vs estado ENSO:")
print(tabla)

### Bloque 8 — Entrenamiento con validación temporal


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


FEATURES_ANOM = [
    "anom_lag1", "anom_lag2", "anom_lag3", "anom_rolling3",
    "oni_lag1",  "oni_lag2",  "oni_lag3",  "oni_rolling3",
    "lst_lag1",  "precip_lag1",
    "mes_sin",   "mes_cos"
]
TARGET_ANOM = "ndvi_anomalia_target"

N_TEST = 12
df_train = df_modelo.iloc[:-N_TEST].copy()
df_test  = df_modelo.iloc[-N_TEST:].copy()

X_train = df_train[FEATURES_ANOM]
y_train = df_train[TARGET_ANOM]
X_test  = df_test[FEATURES_ANOM]
y_test  = df_test[TARGET_ANOM]

rf_anom = RandomForestRegressor(
    n_estimators=300, max_depth=6,
    min_samples_leaf=3, max_features=0.7,
    random_state=42, n_jobs=-1
)
rf_anom.fit(X_train, y_train)
y_pred_anom = rf_anom.predict(X_test)

# Reconstruir NDVI absoluto sumando la climatología
climatologia_test = df_test["ndvi_climatologia"].values
y_pred_ndvi = y_pred_anom + climatologia_test
y_real_ndvi = y_test.values + climatologia_test

print("=== Modelo de Anomalías ===")
print(f"MAE  anomalía: {mean_absolute_error(y_test, y_pred_anom):.4f}")
print(f"RMSE anomalía: {mean_squared_error(y_test, y_pred_anom)**0.5:.4f}")
print(f"R²   anomalía: {r2_score(y_test, y_pred_anom):.4f}")
print()
print(f"MAE  NDVI reconstruido: {mean_absolute_error(y_real_ndvi, y_pred_ndvi):.4f}")
print(f"R²   NDVI reconstruido: {r2_score(y_real_ndvi, y_pred_ndvi):.4f}")

# Comparación mes a mes
comparacion = df_test[["fecha"]].copy()
comparacion["ndvi_real"]     = y_real_ndvi
comparacion["ndvi_predicho"] = y_pred_ndvi
comparacion["anomalia_real"] = y_test.values
comparacion["anomalia_pred"] = y_pred_anom
comparacion["fecha_objetivo"] = comparacion["fecha"] + pd.DateOffset(months=1)
print("\n", comparacion[["fecha_objetivo","ndvi_real","ndvi_predicho","anomalia_real","anomalia_pred"]].to_string())

### Variabilidad

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

FEATURES = [
    "ndvi_lag1", "ndvi_lag2", "ndvi_lag3", "ndvi_rolling3",
    "oni_lag1",  "oni_lag2",  "oni_lag3",  "oni_rolling3",
    "lst_lag1",  "precip_lag1",
    "mes_sin",   "mes_cos"
]
TARGET = "ndvi_target"

tscv = TimeSeriesSplit(n_splits=5)
X_full = df_modelo[FEATURES]
y_full = df_modelo[TARGET]

print("=== Random Forest ===")
scores_rf = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    rf_cv = RandomForestRegressor(
        n_estimators=300, max_depth=8,
        min_samples_leaf=3, max_features=0.7,
        random_state=42, n_jobs=-1
    )
    rf_cv.fit(X_tr, y_tr)
    y_val_pred = rf_cv.predict(X_val)
    
    mae = mean_absolute_error(y_val, y_val_pred)
    r2  = r2_score(y_val, y_val_pred)
    scores_rf.append({"fold": fold+1, "mae": mae, "r2": r2,
        "periodo": f"{df_modelo['fecha'].iloc[val_idx[0]].date()} → {df_modelo['fecha'].iloc[val_idx[-1]].date()}"
    })
    print(f"Fold {fold+1} | MAE: {mae:.4f} | R²: {r2:.4f} | {scores_rf[-1]['periodo']}")

print(f"\nMAE promedio: {np.mean([s['mae'] for s in scores_rf]):.4f} ± {np.std([s['mae'] for s in scores_rf]):.4f}")
print(f"R²  promedio: {np.mean([s['r2']  for s in scores_rf]):.4f} ± {np.std([s['r2']  for s in scores_rf]):.4f}")

print("\n=== Gradient Boosting ===")
scores_gb = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    gb_cv = GradientBoostingRegressor(
        n_estimators=500, max_depth=3,
        learning_rate=0.03, subsample=0.7,
        min_samples_leaf=4, random_state=42
    )
    gb_cv.fit(X_tr, y_tr)
    y_val_pred = gb_cv.predict(X_val)
    
    mae = mean_absolute_error(y_val, y_val_pred)
    r2  = r2_score(y_val, y_val_pred)
    scores_gb.append({"fold": fold+1, "mae": mae, "r2": r2})
    print(f"Fold {fold+1} | MAE: {mae:.4f} | R²: {r2:.4f} | {df_modelo['fecha'].iloc[val_idx[0]].date()} → {df_modelo['fecha'].iloc[val_idx[-1]].date()}")

print(f"\nMAE promedio: {np.mean([s['mae'] for s in scores_gb]):.4f} ± {np.std([s['mae'] for s in scores_gb]):.4f}")
print(f"R²  promedio: {np.mean([s['r2']  for s in scores_gb]):.4f} ± {np.std([s['r2']  for s in scores_gb]):.4f}")

### Guardar Modelo

In [ ]:
import joblib

# Entrenar modelo final con TODOS los datos
rf_final = RandomForestRegressor(
    n_estimators=300, max_depth=6,
    min_samples_leaf=3, max_features=0.7,
    random_state=42, n_jobs=-1
)
FEATURES_FINAL = [
    "anom_lag1", "anom_lag2", "anom_lag3", "anom_rolling3",
    "oni_lag1", "oni_lag2", "oni_lag3", "oni_rolling3",
    "lst_lag1", "precip_lag1", "mes_sin", "mes_cos"
]
rf_final.fit(df_modelo[FEATURES_FINAL], df_modelo["ndvi_anomalia_target"])

# Guardar modelo + climatología (necesaria para reconstruir NDVI absoluto)
payload = {
    "model": rf_final,
    "features": FEATURES_FINAL,
    "climatologia": df_modelo.groupby("mes")["ndvi_climatologia"].mean().to_dict()
}
joblib.dump(payload, "modelo_chagres.pkl")
print("✅ modelo_chagres.pkl guardado")

### Análisis de eventos El Niño intensos vs NDVI

In [ ]:
# Análisis de eventos El Niño intensos vs NDVI
# Definir umbral de El Niño intenso: ONI >= 1.0

df_modelo["nino_intenso"] = (df_modelo["oni_lag1"] >= 1.0).astype(int)
df_modelo["nino_moderado"] = ((df_modelo["oni_lag1"] >= 0.5) & 
                               (df_modelo["oni_lag1"] < 1.0)).astype(int)
df_modelo["nina"] = (df_modelo["oni_lag1"] <= -0.5).astype(int)

print("=== Anomalía NDVI promedio por estado ENSO ===")
print(f"El Niño intenso (ONI≥1.0):  {df_modelo[df_modelo['nino_intenso']==1]['ndvi_anomalia'].mean():+.4f}  (n={df_modelo['nino_intenso'].sum()})")
print(f"El Niño moderado (0.5-1.0): {df_modelo[df_modelo['nino_moderado']==1]['ndvi_anomalia'].mean():+.4f}  (n={df_modelo['nino_moderado'].sum()})")
print(f"Neutro:                      {df_modelo[(df_modelo['nino_intenso']==0) & (df_modelo['nino_moderado']==0) & (df_modelo['nina']==0)]['ndvi_anomalia'].mean():+.4f}")
print(f"La Niña (ONI≤-0.5):         {df_modelo[df_modelo['nina']==1]['ndvi_anomalia'].mean():+.4f}  (n={df_modelo['nina'].sum()})")

# Correlación solo durante eventos intensos
mask_nino = df_modelo["oni_lag1"] >= 1.0
print(f"\nCorrelación ONI-NDVI durante El Niño intenso: r = {df_modelo[mask_nino]['oni_lag1'].corr(df_modelo[mask_nino]['ndvi_anomalia']):.4f}")
print(f"Correlación ONI-NDVI durante todo el período: r = {df_modelo['oni_lag1'].corr(df_modelo['ndvi_anomalia']):.4f}")

# Duración promedio de anomalías negativas consecutivas
df_modelo["estres"] = (df_modelo["ndvi_anomalia"] < -0.02).astype(int)
print(f"\n% meses con anomalía < -0.02 durante El Niño intenso: {df_modelo[mask_nino]['estres'].mean()*100:.1f}%")
print(f"% meses con anomalía < -0.02 en período normal:       {df_modelo[~mask_nino]['estres'].mean()*100:.1f}%")

# Mapas

### Parte 1 — Extracción de NDVI por zonas (grilla) desde GEE

In [ ]:
import ee
import json
import numpy as np
import pandas as pd
from shapely.geometry import shape, box, mapping
from shapely.ops import unary_union
import geopandas as gpd

# ── Cargar geometría del parque ──────────────────────────────────────────────
with open("chagres_geometry.geojson", encoding="utf-8") as f:
    gj = json.load(f)

chagres_shape = shape(gj["features"][0]["geometry"])
bounds = chagres_shape.bounds  # (minx, miny, maxx, maxy)

# ── Crear grilla de celdas dentro del parque ─────────────────────────────────
# ~0.05 grados ≈ 5km, ajusta según qué tan gruesa quieras la grilla
RESOLUCION_GRADOS = 0.05

minx, miny, maxx, maxy = bounds
xs = np.arange(minx, maxx, RESOLUCION_GRADOS)
ys = np.arange(miny, maxy, RESOLUCION_GRADOS)

celdas = []
for i, x in enumerate(xs):
    for j, y in enumerate(ys):
        celda = box(x, y, x + RESOLUCION_GRADOS, y + RESOLUCION_GRADOS)
        # Solo incluir celdas que intersecten con el parque
        if celda.intersects(chagres_shape):
            interseccion = celda.intersection(chagres_shape)
            if interseccion.area > 0:
                celdas.append({
                    "id": f"{i}_{j}",
                    "geometry": interseccion,
                    "centroid_lon": interseccion.centroid.x,
                    "centroid_lat": interseccion.centroid.y
                })

print(f"✅ Grilla generada: {len(celdas)} celdas dentro del Chagres")

# ── Ver la grilla antes de continuar ─────────────────────────────────────────
gdf_grilla = gpd.GeoDataFrame(celdas, geometry="geometry", crs="EPSG:4326")
print(gdf_grilla.head())

### Parte 2 — Extracción NDVI mensual por celda desde GEE

In [ ]:
# ── Extracción NDVI por celda ─────────────────────────────────────────────────
# Ajusta el rango — con 3 años recientes es suficiente para los lags del modelo
FECHA_INICIO_ESPACIAL = "2023-01-01"
FECHA_FIN_ESPACIAL    = "2026-05-01"

def extraer_ndvi_celda(geom_shapely, celda_id):
    """Extrae NDVI mensual promedio para una geometría shapely."""
    geom_ee = ee.Geometry(mapping(geom_shapely))
    
    col = (
        ee.ImageCollection("MODIS/061/MOD13A3")
        .filterBounds(geom_ee)
        .filterDate(FECHA_INICIO_ESPACIAL, FECHA_FIN_ESPACIAL)
        .select("NDVI")
    )

    def reducir(img):
        valor = (
            img.multiply(0.0001)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom_ee,
                scale=1000,
                maxPixels=1e8
            )
            .get("NDVI")
        )
        return ee.Feature(None, {
            "fecha": img.date().format("YYYY-MM-dd"),
            "ndvi": valor,
            "celda_id": celda_id
        })

    feats = col.map(reducir).getInfo()["features"]
    rows = []
    for f in feats:
        p = f["properties"]
        if p["ndvi"] is not None:
            rows.append({
                "celda_id": celda_id,
                "fecha": pd.Timestamp(p["fecha"]),
                "ndvi": p["ndvi"]
            })
    return rows

# Correr para todas las celdas (muestra progreso)
todos_los_registros = []
for idx, celda in enumerate(celdas):
    if idx % 5 == 0:
        print(f"  Procesando celda {idx+1}/{len(celdas)}...")
    registros = extraer_ndvi_celda(celda["geometry"], celda["id"])
    todos_los_registros.extend(registros)

df_espacial = pd.DataFrame(todos_los_registros)
df_espacial["fecha"] = df_espacial["fecha"].dt.to_period("M").dt.to_timestamp()
df_espacial.to_csv("ndvi_espacial_chagres.csv", index=False)
print(f"✅ {len(df_espacial)} registros guardados en ndvi_espacial_chagres.csv")
print(df_espacial.head(10))

### Parte 3 — Construir features por celda y aplicar el modelo


In [ ]:
import joblib

# ── Cargar modelo y climatología ──────────────────────────────────────────────
payload      = joblib.load("modelo_chagres.pkl")
rf_final     = payload["model"]
FEATURES_FINAL = payload["features"]
climatologia = payload["climatologia"]  # dict {mes: ndvi_climatologia}

# ── Cargar datos espaciales y datos globales de ONI/LST/precip ───────────────
df_espacial = pd.read_csv("ndvi_espacial_chagres.csv", parse_dates=["fecha"])
df_oni      = pd.read_csv("oni_mensual_2000.csv",       parse_dates=["fecha"])
df_lst      = pd.read_csv("lst_mensual_chagres_2000.csv", parse_dates=["fecha"])
df_precip   = pd.read_csv("precip_mensual_chagres_2000.csv", parse_dates=["fecha"])

# Normalizar fechas
for df in [df_espacial, df_oni, df_lst, df_precip]:
    df["fecha"] = df["fecha"].dt.to_period("M").dt.to_timestamp()

# ONI, LST y precip son globales para el parque (mismo valor para todas las celdas)
df_global = (
    df_oni
    .merge(df_lst,    on="fecha", how="inner")
    .merge(df_precip, on="fecha", how="inner")
    .sort_values("fecha")
    .reset_index(drop=True)
)

# ── Función para construir features de una celda ──────────────────────────────
def construir_features_celda(df_ndvi_celda, df_global, climatologia):
    """
    Recibe el historial NDVI de una celda y construye el mismo
    feature engineering que usaste en el entrenamiento.
    """
    df = df_ndvi_celda.sort_values("fecha").copy()
    df = df.merge(df_global, on="fecha", how="inner")

    # Climatología (promedio histórico por mes para ESTA celda)
    df["mes"] = df["fecha"].dt.month
    clim_celda = df.groupby("mes")["ndvi"].mean().to_dict()
    # Si algún mes no tiene datos propios, usar la climatología global
    df["ndvi_climatologia"] = df["mes"].map(
        lambda m: clim_celda.get(m, climatologia.get(m, df["ndvi"].mean()))
    )

    df["ndvi_anomalia"] = df["ndvi"] - df["ndvi_climatologia"]

    # Lags (igual que en entrenamiento)
    df["anom_lag1"]     = df["ndvi_anomalia"].shift(1)
    df["anom_lag2"]     = df["ndvi_anomalia"].shift(2)
    df["anom_lag3"]     = df["ndvi_anomalia"].shift(3)
    df["anom_rolling3"] = df["ndvi_anomalia"].shift(1).rolling(3).mean()

    df["oni_lag1"]      = df["oni"].shift(1)
    df["oni_lag2"]      = df["oni"].shift(2)
    df["oni_lag3"]      = df["oni"].shift(3)
    df["oni_rolling3"]  = df["oni"].shift(1).rolling(3).mean()

    df["lst_lag1"]      = df["lst_celsius"].shift(1)
    df["precip_lag1"]   = df["precip_mm"].shift(1)

    df["mes_sin"]       = np.sin(2 * np.pi * df["mes"] / 12)
    df["mes_cos"]       = np.cos(2 * np.pi * df["mes"] / 12)

    return df.dropna().reset_index(drop=True)

### Parte 4 — Predicción para meses futuros (Jul-Dic 2026)


In [ ]:
# ── Meses futuros a predecir ──────────────────────────────────────────────────
MESES_FUTUROS = pd.date_range("2026-07-01", "2026-12-01", freq="MS")

# ONI proyectado para meses futuros (usa el último conocido como aproximación)
# Puedes reemplazar estos valores con el pronóstico oficial de NOAA si lo tienes
oni_futuro = {
    pd.Timestamp("2026-07-01"): -0.3,
    pd.Timestamp("2026-08-01"): -0.4,
    pd.Timestamp("2026-09-01"): -0.4,
    pd.Timestamp("2026-10-01"): -0.3,
    pd.Timestamp("2026-11-01"): -0.2,
    pd.Timestamp("2026-12-01"): -0.1,
}

# ── Generar predicciones por celda y mes ─────────────────────────────────────
resultados = []  # lista de dicts {celda_id, fecha, prob_estres, ndvi_pred}

for celda in celdas:
    cid = celda["id"]
    df_celda = df_espacial[df_espacial["celda_id"] == cid].copy()

    if len(df_celda) < 6:
        # Pocas observaciones, saltar
        continue

    # Construir features históricos
    df_feat = construir_features_celda(df_celda, df_global, climatologia)

    if df_feat.empty:
        continue

    # Climatología de esta celda por mes
    clim_celda = df_feat.groupby("mes")["ndvi_climatologia"].mean().to_dict()

    # Tomar los últimos valores conocidos (fila más reciente = May 2026 o lo último)
    ultima = df_feat.iloc[-1]

    for mes_fut in MESES_FUTUROS:
        mes_num = mes_fut.month

        # Construir el vector de features para este mes futuro
        # Los lags se desplazan: lo que era lag1 ahora es lag2, etc.
        # Para simplicidad usamos la última fila disponible + actualizamos mes
        features_row = {
            "anom_lag1":     ultima["ndvi_anomalia"],       # anomalía del mes anterior conocido
            "anom_lag2":     ultima["anom_lag1"],
            "anom_lag3":     ultima["anom_lag2"],
            "anom_rolling3": (ultima["ndvi_anomalia"] + ultima["anom_lag1"] + ultima["anom_lag2"]) / 3,
            "oni_lag1":      oni_futuro.get(mes_fut, ultima["oni"]),
            "oni_lag2":      ultima["oni_lag1"],
            "oni_lag3":      ultima["oni_lag2"],
            "oni_rolling3":  (oni_futuro.get(mes_fut, ultima["oni"]) + ultima["oni_lag1"] + ultima["oni_lag2"]) / 3,
            "lst_lag1":      ultima["lst_celsius"],
            "precip_lag1":   ultima["precip_mm"],
            "mes_sin":       np.sin(2 * np.pi * mes_num / 12),
            "mes_cos":       np.cos(2 * np.pi * mes_num / 12),
        }

        X_pred = pd.DataFrame([features_row])[FEATURES_FINAL]
        anom_pred = rf_final.predict(X_pred)[0]

        # NDVI absoluto predicho
        ndvi_clim = clim_celda.get(mes_num, climatologia.get(mes_num, 0.6))
        ndvi_pred = anom_pred + ndvi_clim

        # Probabilidad de estrés: usamos los árboles individuales del RF
        # para estimar incertidumbre (% de árboles que predicen anomalía negativa)
        predicciones_arboles = np.array([
            tree.predict(X_pred)[0] for tree in rf_final.estimators_
        ])
        prob_estres = (predicciones_arboles < 0).mean()  # % árboles que ven estrés

        resultados.append({
            "celda_id":   cid,
            "fecha":      mes_fut,
            "ndvi_pred":  round(ndvi_pred, 4),
            "anom_pred":  round(anom_pred, 4),
            "prob_estres": round(prob_estres, 3),
        })

df_pred = pd.DataFrame(resultados)
print(f"✅ {len(df_pred)} predicciones generadas")
print(df_pred.head(12))

### Parte 5 — Exportar GeoJSON por mes (los archivos que usará la web)


In [ ]:
import os
os.makedirs("predicciones_map", exist_ok=True)

def nivel_riesgo(prob):
    if pd.isna(prob):   return "sin_datos"
    if prob >= 0.70:    return "alto"
    if prob >= 0.45:    return "moderado"
    return "bajo"

for mes_fut in MESES_FUTUROS:
    df_mes = df_pred[df_pred["fecha"] == mes_fut].copy()
    df_mes = df_mes.rename(columns={"celda_id": "id"})

    gdf_mes = gdf_grilla.merge(df_mes, on="id", how="left")
    gdf_mes["riesgo"] = gdf_mes["prob_estres"].apply(nivel_riesgo)

    nombre = f"predicciones_map/pred_{mes_fut.strftime('%Y_%m')}.geojson"
    gdf_mes[["id", "geometry", "ndvi_pred", "anom_pred",
             "prob_estres", "riesgo"]].to_file(nombre, driver="GeoJSON")
    print(f"✅ {nombre} — {len(gdf_mes)} zonas")

print("\n🗺️ Todos los GeoJSON generados en /predicciones_map/")